# ✂️ EWIT – Cắt video Zoom theo công thức AI
Chạy **lần lượt từng ô** bằng nút ▶ bên trái mỗi ô (hoặc Shift+Enter). Ô nào có chữ ✏️ thì có thể sửa giá trị trước khi chạy.

## Bước 1 – Kết nối Google Drive và tải tool
Sẽ có cửa sổ hỏi quyền truy cập Drive → chọn tài khoản → **Cho phép**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import subprocess, sys, shutil
subprocess.run('rm -rf /content/VideoApps && git clone -q --depth 1 -b claude/blissful-thompson-ey00yo '
               'https://github.com/tnho113/VideoApps /content/VideoApps', shell=True, check=True)
TOOL = '/content/VideoApps/tools/zoom_cut'
sys.path.insert(0, TOOL)

def run(*args):
    """Chạy 1 script của tool và in kết quả; dừng nếu lỗi."""
    r = subprocess.run([sys.executable, *args], cwd=TOOL, capture_output=True, text=True)
    print(r.stdout[-6000:])
    if r.returncode:
        print(r.stderr[-3000:]); raise RuntimeError('Lỗi khi chạy ' + args[0])

print('✅ Đã kết nối Drive và tải tool')

## Bước 2 – Chọn thư mục chứa video ✏️
Để trống `FOLDER` thì tool tự tìm thư mục tên `TEN_THU_MUC` trong Drive của bạn.

In [ ]:
TEN_THU_MUC = 'IT Team Meetings'   # ✏️
FOLDER = ''                         # ✏️ hoặc dán đường dẫn đầy đủ, ví dụ '/content/drive/MyDrive/IT Team Meetings'
import glob, os
if not FOLDER:
    hits = [p for d in range(0, 4) for p in glob.glob('/content/drive/MyDrive/' + '*/' * d + TEN_THU_MUC)]
    assert hits, 'Không tìm thấy thư mục, hãy dán đường dẫn vào FOLDER'
    FOLDER = hits[0]
VIDEO = [f for f in glob.glob(FOLDER + '/*.mp4') if '_da_cat' not in f][0]
VTT = glob.glob(FOLDER + '/*.vtt')[0]
RECIPES = sorted(glob.glob(FOLDER + '/cong_thuc*.json'))
OUT = FOLDER + '/ket_qua'; os.makedirs(OUT, exist_ok=True)
print('Thư mục :', FOLDER); print('Video   :', os.path.basename(VIDEO)); print('Transcript:', os.path.basename(VTT))
print('Công thức cắt:', [os.path.basename(r) for r in RECIPES])

## Bước 3 – Xem trước các đoạn sẽ cắt ✏️
`CHAY_THU = True` chỉ xử lý 10 phút đầu. Đọc kỹ báo cáo: mỗi đoạn có lý do và lời thoại bị bỏ.

In [ ]:
CHAY_THU = True        # ✏️ True = 10 phút đầu; False = cả video
RECIPE = RECIPES[-1]   # ✏️ công thức dùng (mặc định file mới nhất)
UNTIL = ['--until', '00:10:00'] if CHAY_THU else []
TEN = 'thu_10phut' if CHAY_THU else 'video_da_cat'
run('cut_video.py', VIDEO, RECIPE, '--vtt', VTT, *UNTIL, '--dry-run')

## Bước 4 – Cắt video
Bản thử 10 phút: vài phút. Cả video 150 phút: có thể 1–2 giờ, **giữ tab mở**.

In [ ]:
run('cut_video.py', VIDEO, RECIPE, '--vtt', VTT, *UNTIL, '-o', f'/content/{TEN}.mp4')
for ext in ('mp4', 'srt'):
    shutil.copy(f'/content/{TEN}.{ext}', OUT)
print('✅ Đã lưu vào Drive:', OUT)

## Bước 5 – Tạo thumbnail ✏️
Mặc định dùng **mẫu Training (Mạch lá)**: chữ lấy từ công thức cắt, sửa trực tiếp các dòng ✏️ nếu cần rồi chạy lại ô.
Đặt `MAU = ''` để dùng kiểu cũ (khung hình video + chữ).

In [ ]:
import json, re, glob
r = json.loads(re.search(r'\{.*\}', open(RECIPE, encoding='utf-8').read(), re.S).group(0))
t = r.get('thumbnail', {})
MAU       = t.get('mau', 'training')               # ✏️ 'training' hoặc '' (khung hình video)
DONG_TREN = t.get('dong_tren', '// TRAINING')      # ✏️
TIEU_DE   = t.get('tieu_de', t.get('chu', 'EWIT')) # ✏️
NGUOI     = t.get('nguoi', '')                     # ✏️ ví dụ 'Chia sẻ: Trần Văn Thừa'
NGAY      = t.get('ngay', '')                      # ✏️ ví dụ 'Thứ Hai · 21:00 · 09/11/2026'
THOI_DIEM = '00:01:00'                             # ✏️ chỉ dùng khi MAU = ''
if MAU:
    nen = json.load(open(f'{TOOL}/mau/{MAU}.json', encoding='utf-8'))['nen']
    hits = [p for d in range(6) for p in glob.glob('/content/drive/MyDrive/' + '*/' * d + nen)]
    assert hits, f'Không tìm thấy ảnh nền {nen} trên Drive'
    run('make_thumbnail.py', '-', '--mau', f'mau/{MAU}.json', '--nen', hits[0], '--dong-tren', DONG_TREN,
        '--text', TIEU_DE, '--nguoi', NGUOI, '--ngay', NGAY, '-o', f'{OUT}/{TEN}.jpg')
else:
    run('make_thumbnail.py', f'/content/{TEN}.mp4', '--time', THOI_DIEM, '--text', TIEU_DE, '-o', f'{OUT}/{TEN}.jpg')
from IPython.display import Image, display; display(Image(f'{OUT}/{TEN}.jpg', width=640))

## Bước 6 – Xem video đã cắt
Mở thư mục `ket_qua` trong Drive để xem video, phụ đề `.srt` và thumbnail. Ổn thì đổi `CHAY_THU = False` ở Bước 3 rồi chạy lại Bước 3 → 5.

## Bước 7 – Đăng YouTube (làm sau khi bản cắt đã ổn)
Cần file `client_secret.json` trong thư mục – xem hướng dẫn trong `tools/zoom_cut/README.md`.

In [ ]:
DANG_YOUTUBE = False   # ✏️ đổi thành True khi đã sẵn sàng
if DANG_YOUTUBE:
    from upload_youtube import get_credentials, upload
    creds = get_credentials(FOLDER + '/client_secret.json', FOLDER + '/youtube_token.json')
    upload(f'/content/{TEN}.mp4', r.get('tieu_de', TEN), r.get('mo_ta', ''), r.get('the_tag', []),
           f'{OUT}/{TEN}.jpg', 'private', creds, f'/content/{TEN}.srt')